In [1]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.chart import BarChart, Reference


def generate_item_wise_report(input_csv, output_xlsx, date_label):
    df = pd.read_csv(input_csv)

    HEADER_BLUE = "FF1F4E78"
    SECTION_BLUE = "FF2E75B6"
    WHITE_BOLD = Font(bold=True, color="FFFFFFFF", size=10)
    TITLE_FONT = Font(bold=True, color="FFFFFFFF", size=16)
    SUBTITLE_FONT = Font(size=10, color="FFFFFFFF")
    SECTION_FONT = Font(bold=True, size=12, color="FFFFFFFF")

    def fill(c):
        return PatternFill("solid", fgColor=c)

    wb = Workbook()
    ws1 = wb.active
    ws1.title = "Category & Restaurant Pivot"

    # ================= SECTION A: CATEGORY SUMMARY =================
    agg_cols = ["Quantity", "Amount", "Offline Quantity", "Offline Amount",
                "Online Quantity", "Online Amount", "Zomato Quantity", "Zomato Amount",
                "Swiggy Quantity", "Swiggy Amount"]
    cat_summary = df.groupby("Category", as_index=False)[agg_cols].sum()
    cat_summary = cat_summary.sort_values("Amount", ascending=False).reset_index(drop=True)
    grand_amount = cat_summary["Amount"].sum()
    cat_summary["% of Total Sales"] = cat_summary["Amount"] / grand_amount

    ws1.merge_cells("A1:O1")
    c = ws1["A1"]
    c.value = f"APSARA ITEM-WISE SALES REPORT — {date_label.upper()}"
    c.font = TITLE_FONT; c.fill = fill(HEADER_BLUE)
    c.alignment = Alignment(horizontal="left", vertical="center")
    ws1.row_dimensions[1].height = 26

    ws1.merge_cells("A2:O2")
    c = ws1["A2"]
    c.value = "One consolidated view — Category Summary, Restaurant x Category Pivot, and Item Performance. All figures in ₹."
    c.font = SUBTITLE_FONT; c.fill = fill(HEADER_BLUE)

    ws1.merge_cells("A4:O4")
    c = ws1["A4"]
    c.value = "SECTION A — CATEGORY SUMMARY"
    c.font = SECTION_FONT; c.fill = fill(SECTION_BLUE)

    HEADER_ROW_A = 5
    headers_A = ["Category", "Quantity", "Amount", "Offline Quantity", "Offline Amount",
                 "Online Quantity", "Online Amount", "Zomato Quantity", "Zomato Amount",
                 "Swiggy Quantity", "Swiggy Amount", "% of Total Sales"]
    for i, h in enumerate(headers_A, start=1):
        cell = ws1.cell(row=HEADER_ROW_A, column=i, value=h)
        cell.font = WHITE_BOLD; cell.fill = fill(HEADER_BLUE)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    for r_idx, row in enumerate(cat_summary.itertuples(index=False), start=HEADER_ROW_A + 1):
        for c_idx, val in enumerate(row, start=1):
            cell = ws1.cell(row=r_idx, column=c_idx, value=val)
            if c_idx == 12:
                cell.number_format = "0.0%"
            elif c_idx >= 2:
                cell.number_format = "#,##0"

    cat_total_row = HEADER_ROW_A + 1 + len(cat_summary)
    ws1.cell(row=cat_total_row, column=1, value="GRAND TOTAL").font = Font(bold=True)
    for c_idx in range(2, 12):
        col_letter = get_column_letter(c_idx)
        f = ws1.cell(row=cat_total_row, column=c_idx,
                      value=f"=SUM({col_letter}{HEADER_ROW_A+1}:{col_letter}{cat_total_row-1})")
        f.font = Font(bold=True); f.number_format = "#,##0"
    pct_total = ws1.cell(row=cat_total_row, column=12, value=1)
    pct_total.number_format = "0.0%"; pct_total.font = Font(bold=True)

    ws1.conditional_formatting.add(f"C{HEADER_ROW_A+1}:C{cat_total_row-1}", ColorScaleRule(
        start_type="min", start_color="FFFFFFFF", end_type="max", end_color="4CAF50"))

    # ================= SECTION B: RESTAURANT x CATEGORY PIVOT =================
    category_order = cat_summary["Category"].tolist()
    pivot = df.pivot_table(index="Restaurant", columns="Category", values="Amount",
                            aggfunc="sum", fill_value=0)
    pivot = pivot.reindex(columns=category_order, fill_value=0)
    pivot["Total"] = pivot.sum(axis=1)
    pivot = pivot.sort_values("Total", ascending=False).reset_index()

    SECTION_B_LABEL_ROW = cat_total_row + 3
    ws1.merge_cells(start_row=SECTION_B_LABEL_ROW, start_column=1,
                     end_row=SECTION_B_LABEL_ROW, end_column=15)
    c = ws1.cell(row=SECTION_B_LABEL_ROW, column=1,
                 value="SECTION B — RESTAURANT x CATEGORY SALES PIVOT (colour = sales intensity)")
    c.font = SECTION_FONT; c.fill = fill(SECTION_BLUE)

    HEADER_ROW_B = SECTION_B_LABEL_ROW + 1
    headers_B = ["Restaurant"] + category_order + ["Total"]
    for i, h in enumerate(headers_B, start=1):
        cell = ws1.cell(row=HEADER_ROW_B, column=i, value=h)
        cell.font = WHITE_BOLD; cell.fill = fill(HEADER_BLUE)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    for r_idx, row in enumerate(pivot.itertuples(index=False), start=HEADER_ROW_B + 1):
        for c_idx, val in enumerate(row, start=1):
            cell = ws1.cell(row=r_idx, column=c_idx, value=val)
            if c_idx >= 2:
                cell.number_format = "#,##0"

    pivot_total_row = HEADER_ROW_B + 1 + len(pivot)
    ws1.cell(row=pivot_total_row, column=1, value="GRAND TOTAL").font = Font(bold=True)
    for c_idx in range(2, len(headers_B) + 1):
        col_letter = get_column_letter(c_idx)
        f = ws1.cell(row=pivot_total_row, column=c_idx,
                      value=f"=SUM({col_letter}{HEADER_ROW_B+1}:{col_letter}{pivot_total_row-1})")
        f.font = Font(bold=True); f.number_format = "#,##0"

    last_cat_col = len(category_order) + 1
    for col_idx in range(2, last_cat_col + 1):
        col_letter = get_column_letter(col_idx)
        rng = f"{col_letter}{HEADER_ROW_B+1}:{col_letter}{pivot_total_row-1}"
        ws1.conditional_formatting.add(rng, ColorScaleRule(
            start_type="min", start_color="FFFFFFFF",
            mid_type="percentile", mid_value=50, mid_color="FFF3B0",
            end_type="max", end_color="ED7D31"))
    total_col_letter = get_column_letter(last_cat_col + 1)
    ws1.conditional_formatting.add(
        f"{total_col_letter}{HEADER_ROW_B+1}:{total_col_letter}{pivot_total_row-1}",
        ColorScaleRule(start_type="min", start_color="F8696B",
                        mid_type="percentile", mid_value=50, mid_color="FFEB84",
                        end_type="max", end_color="63BE7B"))

    ws1.freeze_panes = "B5"
    ws1.column_dimensions["A"].width = 32
    ws1.column_dimensions["B"].width = 14
    for i in range(3, len(headers_B) + 1):
        ws1.column_dimensions[get_column_letter(i)].width = 14

    # ================= SHEET 2: TOP 15 & BOTTOM 15 ITEMS =================
    ws2 = wb.create_sheet("Item Performance")
    item_summary = df.groupby(["Item Name", "Category"], as_index=False).agg(
        Quantity=("Quantity", "sum"),
        Amount=("Amount", "sum"),
        Offline_Amount=("Offline Amount", "sum"),
        Online_Amount=("Online Amount", "sum"),
        Zomato_Amount=("Zomato Amount", "sum"),
        Swiggy_Amount=("Swiggy Amount", "sum"),
    )
    item_summary = item_summary[["Item Name", "Category", "Quantity", "Amount",
                                   "Offline_Amount", "Online_Amount", "Zomato_Amount", "Swiggy_Amount"]]
    item_summary = item_summary.sort_values("Amount", ascending=False).reset_index(drop=True)
    item_summary.insert(0, "Rank", range(1, len(item_summary) + 1))

    top15 = item_summary.head(15).reset_index(drop=True)
    bottom15 = item_summary.tail(15).sort_values("Amount", ascending=True).reset_index(drop=True)

    ws2.merge_cells("A1:I1")
    c = ws2["A1"]
    c.value = f"APSARA ITEM-WISE SALES REPORT — {date_label.upper()}"
    c.font = TITLE_FONT; c.fill = fill(HEADER_BLUE)
    ws2.row_dimensions[1].height = 26

    ws2.merge_cells("A2:I2")
    c = ws2["A2"]
    c.value = "Section C — Top 15 & Bottom 15 Items by Sales Amount"
    c.font = SUBTITLE_FONT; c.fill = fill(HEADER_BLUE)

    headers_C = ["Rank", "Item Name", "Category", "Quantity", "Amount",
                 "Offline Amount", "Online Amount", "Zomato Amount", "Swiggy Amount"]

    def write_block(label_row, label_text, label_color, data_df):
        ws2.cell(row=label_row, column=1, value=label_text).font = Font(bold=True, size=13, color=label_color)
        header_row = label_row + 1
        for i, h in enumerate(headers_C, start=1):
            cell = ws2.cell(row=header_row, column=i, value=h)
            cell.font = WHITE_BOLD; cell.fill = fill(HEADER_BLUE)
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        row_fill = fill("E2EFDA") if label_color == "1F4E78" else fill("FCE4E4")
        r = header_row + 1
        for row in data_df.itertuples(index=False):
            for c_idx, val in enumerate(row, start=1):
                cell = ws2.cell(row=r, column=c_idx, value=val)
                cell.fill = row_fill
                if c_idx >= 4:
                    cell.number_format = "#,##0"
            r += 1
        return header_row, r

    TOP_LABEL_ROW = 4
    top_header_row, next_free = write_block(TOP_LABEL_ROW, "TOP 15 ITEMS", "1F4E78", top15)

    BOTTOM_LABEL_ROW = next_free + 1
    write_block(BOTTOM_LABEL_ROW, "BOTTOM 15 ITEMS", "C00000", bottom15)

    chart2 = BarChart()
    chart2.type = "col"
    chart2.title = "Top 15 Items by Sales Amount (₹)"
    chart2.style = 11
    data_ref2 = Reference(ws2, min_col=5, min_row=top_header_row, max_row=top_header_row + 15)
    cats_ref2 = Reference(ws2, min_col=2, min_row=top_header_row + 1, max_row=top_header_row + 15)
    chart2.add_data(data_ref2, titles_from_data=True)
    chart2.set_categories(cats_ref2)
    chart2.width, chart2.height = 20, 10
    ws2.add_chart(chart2, "K4")

    ws2.freeze_panes = "A5"
    ws2.column_dimensions["A"].width = 8
    ws2.column_dimensions["B"].width = 34
    ws2.column_dimensions["C"].width = 20
    ws2.column_dimensions["D"].width = 15

    # ================= SHEET 3: RAW DATA =================
    ws3 = wb.create_sheet("Raw Data")
    raw_cols = list(df.columns)
    for i, h in enumerate(raw_cols, start=1):
        cell = ws3.cell(row=1, column=i, value=h)
        cell.font = WHITE_BOLD; cell.fill = fill(HEADER_BLUE)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for r_idx, row in enumerate(df.itertuples(index=False), start=2):
        for c_idx, val in enumerate(row, start=1):
            ws3.cell(row=r_idx, column=c_idx, value=val)
    ws3.freeze_panes = "A2"
    ws3.column_dimensions["A"].width = 32
    ws3.column_dimensions["B"].width = 14

    wb.save(output_xlsx)
    print("Saved:", output_xlsx)

In [2]:
# Daily
generate_item_wise_report(
    input_csv="apsara_sales_reports_Item Wise Sales Report_Table (2).csv",
    output_xlsx="Apsara_ItemWise_Report_26November2026.xlsx",
    date_label="26th November 2026"
)

Saved: Apsara_ItemWise_Report_26November2026.xlsx
